In [2]:

import math
from collections import Counter, defaultdict

# ---------------------------------
# 1. Training data
# ---------------------------------

training_data = [
    (["Deepak", "studies", "AI"], ["NNP", "VB", "NN"]),
    (["Students", "study", "AI"], ["NNS", "VBP", "NN"]),
    (["Deepak", "studies", "NLP"], ["NNP", "VB", "NN"]),
    (["Students", "study", "NLP"], ["NNS", "VBP", "NN"]),
]

# ---------------------------------
# 2. Initialize counts
# ---------------------------------

tags = sorted({
    tag
    for words, labels in training_data
    for tag in labels
})

vocab = {
    word
    for words, labels in training_data
    for word in words
}

UNK = "<UNK>"
vocab.add(UNK)

start_counts = Counter()
transition_counts = defaultdict(Counter)
emission_counts = defaultdict(Counter)
tag_counts = Counter()

# ---------------------------------
# 3. Count transitions and emissions
# ---------------------------------

for words, labels in training_data:

    start_counts[labels[0]] += 1

    for word, tag in zip(words, labels):
        emission_counts[tag][word] += 1
        tag_counts[tag] += 1

    for previous_tag, current_tag in zip(
        labels[:-1], labels[1:]
    ):
        transition_counts[previous_tag][current_tag] += 1

# ---------------------------------
# 4. Calculate probabilities
# Add-one (Laplace) smoothing
# ---------------------------------

alpha = 1.0
num_tags = len(tags)
num_words = len(vocab)
num_sentences = len(training_data)

start_prob = {}

for tag in tags:
    start_prob[tag] = (
        start_counts[tag] + alpha
    ) / (num_sentences + alpha * num_tags)

transition_prob = {}

for previous_tag in tags:
    transition_prob[previous_tag] = {}

    for current_tag in tags:
        transition_prob[previous_tag][current_tag] = (
            transition_counts[previous_tag][current_tag]
            + alpha
        ) / (
            sum(transition_counts[previous_tag].values())
            + alpha * num_tags
        )

emission_prob = {}

for tag in tags:
    emission_prob[tag] = {}

    for word in vocab:
        emission_prob[tag][word] = (
            emission_counts[tag][word] + alpha
        ) / (
            tag_counts[tag] + alpha * num_words
        )

# ---------------------------------
# 5. Viterbi decoding
# ---------------------------------

def predict_hmm(sentence):

    words = [
        word if word in vocab else UNK
        for word in sentence
    ]

    # Store the best log-score for each tag
    # at each position.
    scores = []
    backpointers = []

    first_scores = {}

    for tag in tags:
        first_scores[tag] = (
            math.log(start_prob[tag])
            + math.log(emission_prob[tag][words[0]])
        )

    scores.append(first_scores)

    # Process the remaining words.
    for i in range(1, len(words)):

        current_scores = {}
        current_backpointers = {}

        for current_tag in tags:

            candidates = {
                previous_tag: (
                    scores[i - 1][previous_tag]
                    + math.log(
                        transition_prob[previous_tag][current_tag]
                    )
                    + math.log(
                        emission_prob[current_tag][words[i]]
                    )
                )
                for previous_tag in tags
            }

            best_previous_tag = max(
                candidates, key=candidates.get
            )

            current_scores[current_tag] = (
                candidates[best_previous_tag]
            )

            current_backpointers[current_tag] = (
                best_previous_tag
            )

        scores.append(current_scores)
        backpointers.append(current_backpointers)

    # Find the best final tag.
    best_tag = max(scores[-1], key=scores[-1].get)

    # Backtrack to recover the best sequence.
    predicted_tags = [best_tag]

    for i in range(len(words) - 1, 0, -1):
        best_tag = backpointers[i - 1][best_tag]
        predicted_tags.append(best_tag)

    predicted_tags.reverse()

    return predicted_tags


# ---------------------------------
# 6. Test the model
# ---------------------------------

test_sentence = ["Deepak", "studies", "AI"]

predicted_labels = predict_hmm(test_sentence)

print("Input sentence:", test_sentence)
print("Predicted labels:", predicted_labels)

for word, tag in zip(test_sentence, predicted_labels):
    print(f"{word:12s} -> {tag}")


Input sentence: ['Deepak', 'studies', 'AI']
Predicted labels: ['NNP', 'VB', 'NN']
Deepak       -> NNP
studies      -> VB
AI           -> NN


In [3]:

# =========================================
# 1. Define a Probabilistic Context-Free Grammar
# =========================================

grammar = [
    ("S",   ("NP", "VP"), 1.00),

    ("NP",  ("Det", "N"), 0.70),
    ("NP",  ("students",), 0.15),
    ("NP",  ("AI",), 0.15),

    ("VP",  ("V", "NP"), 1.00),

    ("Det", ("the",), 1.00),
    ("N",   ("student",), 1.00),
    ("V",   ("studies",), 1.00),
]


# =========================================
# 2. Separate lexical and binary rules
# =========================================

lexical_rules = []
binary_rules = []

for lhs, rhs, probability in grammar:
    if len(rhs) == 1 and rhs[0].islower():
        lexical_rules.append((lhs, rhs[0], probability))
    elif len(rhs) == 2:
        binary_rules.append((lhs, rhs[0], rhs[1], probability))
    else:
        # One-word lexical rules such as NP -> AI
        lexical_rules.append((lhs, rhs[0], probability))


# =========================================
# 3. CKY parsing algorithm
# =========================================

def parse_pcfg(sentence):
    words = sentence.split()
    n = len(words)

    if n == 0:
        return None, 0.0

    # chart[i][j] stores the best parse for
    # the words from position i up to, but
    # not including, position j.
    chart = [
        [dict() for _ in range(n + 1)]
        for _ in range(n)
    ]

    # Fill the chart with lexical rules.
    for i, word in enumerate(words):
        for lhs, terminal, probability in lexical_rules:
            if word == terminal:
                chart[i][i + 1][lhs] = (
                    probability,
                    (lhs, word)
                )

    # Consider spans of length 2, 3, ..., n.
    for span_length in range(2, n + 1):
        for i in range(n - span_length + 1):
            j = i + span_length

            # Try every possible split.
            for k in range(i + 1, j):
                left_chart = chart[i][k]
                right_chart = chart[k][j]

                for lhs, left_symbol, right_symbol, rule_prob in binary_rules:

                    if (left_symbol in left_chart
                            and right_symbol in right_chart):

                        left_prob, left_tree = left_chart[left_symbol]
                        right_prob, right_tree = right_chart[right_symbol]

                        # Probability of the complete subtree.
                        probability = (
                            rule_prob * left_prob * right_prob
                        )

                        # Keep the most probable tree for this symbol.
                        if (lhs not in chart[i][j]
                                or probability > chart[i][j][lhs][0]):

                            tree = (lhs, left_tree, right_tree)

                            chart[i][j][lhs] = (
                                probability,
                                tree
                            )

    # The whole sentence must have the start symbol S.
    if "S" not in chart[0][n]:
        return None, 0.0

    probability, tree = chart[0][n]["S"]
    return tree, probability


# =========================================
# 4. Display the parse tree
# =========================================

def print_tree(tree, level=0):
    if len(tree) == 2 and isinstance(tree[1], str):
        print("  " * level + f"{tree[0]} -> {tree[1]}")
    else:
        print("  " * level + tree[0])
        for child in tree[1:]:
            print_tree(child, level + 1)


# =========================================
# 5. Test the grammar
# =========================================

sentence = "the student studies AI"

tree, probability = parse_pcfg(sentence)

print("Sentence:", sentence)

if tree is not None:
    print("\nMost probable parse tree:")
    print_tree(tree)
    print("\nParse probability:", probability)
else:
    print("\nNo valid parse found by this grammar.")


Sentence: the student studies AI

Most probable parse tree:
S
  NP
    Det -> the
    N -> student
  VP
    V -> studies
    NP -> AI

Parse probability: 0.105


In [4]:

import math
from collections import defaultdict

from sklearn.feature_extraction import DictVectorizer
from sklearn.linear_model import LogisticRegression

# =========================================
# 1. Training data
# =========================================

training_data = [
    (["Deepak", "studies", "AI"], ["NNP", "VB", "NN"]),
    (["Students", "study", "AI"], ["NNS", "VBP", "NN"]),
    (["Deepak", "studies", "NLP"], ["NNP", "VB", "NN"]),
    (["Students", "study", "NLP"], ["NNS", "VBP", "NN"]),
]

tags = sorted({
    tag
    for words, labels in training_data
    for tag in labels
})

START = "<START>"
UNK = "<UNK>"


# =========================================
# 2. Extract features from a word position
# =========================================

def extract_features(words, i, previous_tag):
    word = words[i]

    previous_word = words[i - 1] if i > 0 else "<BOS>"
    next_word = words[i + 1] if i < len(words) - 1 else "<EOS>"

    return {
        "previous_tag": previous_tag,
        "word": word.lower(),
        "previous_word": previous_word.lower(),
        "next_word": next_word.lower(),
        "prefix1": word[:1].lower(),
        "prefix2": word[:2].lower(),
        "suffix1": word[-1:].lower(),
        "suffix2": word[-2:].lower(),
        "is_capitalized": word[0].isupper(),
        "is_all_upper": word.isupper(),
        "word_length": min(len(word), 15),
        "position_is_start": i == 0,
        "position_is_end": i == len(words) - 1,
    }


# =========================================
# 3. Prepare supervised training examples
# =========================================

X_train = []
y_train = []

for words, labels in training_data:
    for i, current_tag in enumerate(labels):

        previous_tag = START if i == 0 else labels[i - 1]

        features = extract_features(
            words, i, previous_tag
        )

        X_train.append(features)
        y_train.append(current_tag)


# =========================================
# 4. Convert feature dictionaries to vectors
# =========================================

vectorizer = DictVectorizer(sparse=True)

X_train_vectorized = vectorizer.fit_transform(X_train)


# =========================================
# 5. Train the local conditional classifier
# =========================================

memm = LogisticRegression(
    max_iter=1000,
    C=1.0
)

memm.fit(X_train_vectorized, y_train)


# =========================================
# 6. MEMM decoding using dynamic programming
# =========================================

def predict_memm(sentence):
    if not sentence:
        return []

    n = len(sentence)

    # scores[i][tag] = best log-probability
    # of a path ending with tag at position i.
    scores = []
    backpointers = []

    # First word: previous tag is START.
    first_scores = {}

    features = extract_features(sentence, 0, START)
    vector = vectorizer.transform([features])
    probabilities = memm.predict_proba(vector)[0]

    for tag, probability in zip(memm.classes_, probabilities):
        first_scores[tag] = math.log(max(probability, 1e-12))

    scores.append(first_scores)

    # Remaining words.
    for i in range(1, n):
        current_scores = {}
        current_backpointers = {}

        for current_tag in tags:
            best_score = float("-inf")
            best_previous_tag = None

            for previous_tag in tags:
                features = extract_features(
                    sentence, i, previous_tag
                )

                vector = vectorizer.transform([features])
                probabilities = memm.predict_proba(vector)[0]

                probability_by_tag = dict(
                    zip(memm.classes_, probabilities)
                )

                probability = probability_by_tag[current_tag]

                candidate_score = (
                    scores[i - 1][previous_tag]
                    + math.log(max(probability, 1e-12))
                )

                if candidate_score > best_score:
                    best_score = candidate_score
                    best_previous_tag = previous_tag

            current_scores[current_tag] = best_score
            current_backpointers[current_tag] = best_previous_tag

        scores.append(current_scores)
        backpointers.append(current_backpointers)

    # Select the best final tag.
    best_tag = max(scores[-1], key=scores[-1].get)

    # Recover the sequence by following backpointers.
    predicted_tags = [best_tag]

    for i in range(n - 1, 0, -1):
        best_tag = backpointers[i - 1][best_tag]
        predicted_tags.append(best_tag)

    predicted_tags.reverse()
    return predicted_tags


# =========================================
# 7. Test the model
# =========================================

test_sentence = ["Deepak", "studies", "AI"]

predicted_labels = predict_memm(test_sentence)

print("Input sentence:", test_sentence)
print("Predicted labels:", predicted_labels)

for word, tag in zip(test_sentence, predicted_labels):
    print(f"{word:12s} -> {tag}")


Input sentence: ['Deepak', 'studies', 'AI']
Predicted labels: ['NNP', 'VB', 'NN']
Deepak       -> NNP
studies      -> VB
AI           -> NN
